# 06 — Ensemble & Stacking (Exp 6)
**Purpose:** Build OOF stacking ensemble from top models, generate final submission.

**Approach:** Diversified base models → StandardScale OOF → KernelRidge meta-learner

(Inspired by winners' approach: 27 ETRs + KernelRidge(poly, degree=2))

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')

SEED = 42
WORK_DIR = '/kaggle/working'

import sys
sys.path.insert(0, '/kaggle/input/ey-water-quality-data/src')
from spatial_cv import LeaveStationGroupOut
from ensemble import generate_oof_predictions, generate_test_predictions

In [ ]:
train = pd.read_parquet(f'{WORK_DIR}/train_featured.parquet')
val = pd.read_parquet(f'{WORK_DIR}/val_featured.parquet')
submission_template = pd.read_csv('/kaggle/input/ey-water-quality-data/submission_template.csv')

TARGET_COLS = []  # auto-detect
for col in train.columns:
    cl = col.lower()
    if any(k in cl for k in ['alkalinity', 'conductance', 'phosphorus']):
        TARGET_COLS.append(col)

STATION_COL = 'station_id'
META_COLS = [STATION_COL, 'Latitude', 'Longitude', 'Sample Date']

all_features = [c for c in train.select_dtypes(include=[np.number]).columns
                if c not in TARGET_COLS + META_COLS]

cv = LeaveStationGroupOut(n_splits=10, station_col=STATION_COL, random_state=SEED)

print(f'Train: {train.shape}, Val: {val.shape}')
print(f'Features: {len(all_features)}, Targets: {TARGET_COLS}')

## Define Diverse Base Models
Diversity comes from: different algorithms, different hyperparameters, different feature subsets.

In [ ]:
# Load best Optuna params from notebook 05 (if saved)
# Otherwise use reasonable defaults with diversity

def build_base_models(target_short):
    """Build diverse base model set for one target."""
    models = [
        # XGBoost variants
        ('xgb_deep', xgb.XGBRegressor(n_estimators=500, max_depth=8, learning_rate=0.05,
                                       subsample=0.8, colsample_bytree=0.8,
                                       random_state=SEED, n_jobs=-1, verbosity=0)),
        ('xgb_shallow', xgb.XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.1,
                                          subsample=0.9, colsample_bytree=0.7,
                                          random_state=SEED+1, n_jobs=-1, verbosity=0)),
        # LightGBM variants
        ('lgb_deep', lgb.LGBMRegressor(n_estimators=500, max_depth=8, learning_rate=0.05,
                                        subsample=0.8, colsample_bytree=0.8,
                                        random_state=SEED, n_jobs=-1, verbose=-1)),
        ('lgb_shallow', lgb.LGBMRegressor(n_estimators=300, max_depth=4, learning_rate=0.1,
                                           subsample=0.9, colsample_bytree=0.7,
                                           random_state=SEED+1, n_jobs=-1, verbose=-1)),
        # ExtraTrees variants (top model in winners' approach)
        ('etr_200', ExtraTreesRegressor(n_estimators=200, max_depth=55, random_state=SEED, n_jobs=-1)),
        ('etr_300', ExtraTreesRegressor(n_estimators=300, max_depth=40, random_state=SEED+1, n_jobs=-1)),
        ('etr_500', ExtraTreesRegressor(n_estimators=500, max_depth=30, 
                                         max_features=0.7, random_state=SEED+2, n_jobs=-1)),
        # RF for diversity
        ('rf_300', RandomForestRegressor(n_estimators=300, max_depth=20, random_state=SEED, n_jobs=-1)),
    ]
    return models

print(f'Base models per target: {len(build_base_models("test"))}')

## Generate OOF Predictions & Train Meta-Learner (Per Target)

In [ ]:
final_predictions = {}
ensemble_scores = {}

for target in TARGET_COLS:
    target_short = target.split('_')[0][:12]
    print(f'\n{"="*60}')
    print(f'ENSEMBLE: {target_short}')
    print(f'{"="*60}')
    
    valid_mask = train[target].notna()
    X_ens = train.loc[valid_mask].reset_index(drop=True)
    y_ens = train.loc[valid_mask, target].reset_index(drop=True)
    
    base_models = build_base_models(target_short)
    
    # Step 1: Generate OOF predictions
    oof_matrix, fitted_models = generate_oof_predictions(
        models=base_models,
        X=X_ens,
        y=y_ens,
        cv=cv,
        feature_cols=all_features
    )
    
    # Step 2: Score individual base models
    print(f'\n  Base model OOF R² scores:')
    for i, (name, _) in enumerate(base_models):
        score = r2_score(y_ens, oof_matrix[:, i])
        print(f'    {name}: {score:.4f}')
    
    # Step 3: Scale OOF predictions
    scaler = StandardScaler()
    oof_scaled = scaler.fit_transform(oof_matrix)
    
    # Step 4: Train meta-learner
    meta = KernelRidge(alpha=0.00001, kernel='poly', degree=2, coef0=0.25)
    meta.fit(oof_scaled, y_ens)
    
    # Evaluate ensemble
    ensemble_pred = meta.predict(oof_scaled)
    ensemble_r2 = r2_score(y_ens, ensemble_pred)
    print(f'\n  Ensemble (KernelRidge) OOF R²: {ensemble_r2:.4f}')
    
    ensemble_scores[target_short] = ensemble_r2
    
    # Step 5: Generate validation predictions
    X_val_feat = val[all_features].fillna(0)
    test_matrix = generate_test_predictions(fitted_models, val, all_features)
    test_scaled = scaler.transform(test_matrix)
    val_preds = meta.predict(test_scaled)
    
    final_predictions[target] = val_preds

print(f'\n\n{"="*60}')
print(f'ENSEMBLE SUMMARY')
print(f'{"="*60}')
for t, s in ensemble_scores.items():
    print(f'  {t}: OOF R² = {s:.4f}')
print(f'  Mean OOF R² = {np.mean(list(ensemble_scores.values())):.4f}')

## Generate Submission File

In [ ]:
# Create submission dataframe
submission = submission_template.copy()

# Map predictions to submission columns
# Adjust column names to match submission template format
for target_col, preds in final_predictions.items():
    # Find matching submission column
    for sub_col in submission.columns:
        if any(k in sub_col.lower() for k in target_col.lower().split('_')[:2]):
            submission[sub_col] = preds
            print(f'  Mapped {target_col} → {sub_col}')
            break

# Save
submission.to_csv(f'{WORK_DIR}/submission.csv', index=False)
print(f'\n✅ Saved submission.csv: {submission.shape}')
display(submission.head())
display(submission.describe())

## Sanity Checks

In [ ]:
# Check submission format
print('=== Submission Validation ===')
print(f'Shape: {submission.shape} (expected: {submission_template.shape})')
print(f'Columns match: {list(submission.columns) == list(submission_template.columns)}')
print(f'Null values: {submission.isnull().sum().sum()}')
print(f'Negative values: {(submission.select_dtypes(include=[np.number]) < 0).sum().sum()}')

# Prediction range sanity
for col in submission.select_dtypes(include=[np.number]).columns:
    print(f'  {col}: [{submission[col].min():.2f}, {submission[col].max():.2f}]')